# Bronze ingestion of Review as because Reviews CSV have some garbage data

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

In [0]:
reviews_schema = StructType([
    StructField("review_id", StringType()),
    StructField("order_id", StringType()),
    StructField("review_score", IntegerType()),
    StructField("review_comment_title", StringType()),
    StructField("review_comment_message", StringType()),
    StructField("review_creation_date", TimestampType()),
    StructField("review_answer_timestamp", TimestampType()),
    StructField("_corrupt_record", StringType()), 
])

In [0]:
# base_path = "/Volumes/workspace/default/e-commerce"
base_path = "/Volumes/ecommerce_lakehouse/default/source_data_csv"

# /Volumes/ecommerce_lakehouse/default/source_data_csv/olist_order_reviews_dataset.csv


df = (
    spark.read.format("csv")
    .option("header", True)
    .option("multiLine", True)              # records can span physical lines
    .option("quote", '"')                   # " wraps a field
    .option("escape", '"')                  # "" inside a field = one literal "  ← the key fix
    .option("mode", "PERMISSIVE")           # don't crash on bad rows
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(reviews_schema)                 # see below — explicit schema matters here
    .load(f"{base_path}/olist_order_reviews_dataset.csv")
)
df.count()

In [0]:
df.display()

In [0]:

clean = df.filter(col("_corrupt_record").isNull())    
clean.count()


In [0]:
clean = clean.filter(length(col("review_id")) == 32)              # valid id shape
clean.count()

In [0]:
clean = clean.drop("_corrupt_record")

In [0]:
clean.limit(5).display()

### Write to Bronze Table

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_lakehouse.bronze.reviews")

In [0]:
spark.table("ecommerce_lakehouse.bronze.reviews").limit(5).display()